In [1]:
# download the model & install biopython (takes a few minutes)
#wget 'https://zenodo.org/records/10515367/files/RBPdetect_v3_ESMfineT33.zip?download=1' -O 'RBPdetect_v3_ESMfine.zip'
!unzip /content/RBPdetect_v3_ESMfine.zip -d /content/RBPdetect_v3_ESMfine
!pip install biopython

unzip:  cannot find or open /content/RBPdetect_v3_ESMfine.zip, /content/RBPdetect_v3_ESMfine.zip.zip or /content/RBPdetect_v3_ESMfine.zip.ZIP.


In [2]:
!module load CUDA

In [3]:
# load libraries
import os
import torch
import pandas as pd
import numpy as np
from Bio import SeqIO
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [4]:
# initiation the model
model_path = 'RBPdetect_v3_ESMfineT33'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval().cuda()

EsmForSequenceClassification(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 1280, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,

In [5]:
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [9]:
# set the name of your FASTA file
fasta_name = '../../PROCESSING/genome-annotate/saurus_update2-pharokka/phanotate.faa'

In [10]:
# make predictions
sequences = [str(record.seq) for record in SeqIO.parse('../genome/'+fasta_name, 'fasta')]
names = [record.id for record in SeqIO.parse('../genome/'+fasta_name, 'fasta')]

predictions = []
scores = []
batch_size = 16
for sequence in tqdm(sequences):
    encoding = tokenizer(sequence, return_tensors="pt", truncation=True, max_length=512).to('cuda:0')
    with torch.no_grad():
        output = model(**encoding)
        predictions.append(int(output.logits.argmax(-1)))
        scores.append(float(output.logits.softmax(-1)[:, 1]))

  0%|          | 0/86 [00:00<?, ?it/s]

In [11]:
# save the results
results = pd.concat([pd.DataFrame(names, columns=['protein_name']),
                         pd.DataFrame(predictions, columns=['preds']),
                        pd.DataFrame(scores, columns=['score'])], axis=1)
results.to_csv('predictions/saurus_predictions.csv', index=False)